# Modelo de lenguaje (zero-shot)

Descubre seis categorías y clasifica una muestra de 2.000 reseñas con Llama 3.1. La ejecución completa requiere Ollama y demora unas tres horas.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Muestreo, descubrimiento de categorías y clasificación (Tabla 4.3)

In [ ]:
import json, re, time, os
import pandas as pd
import numpy as np
from tqdm import tqdm
import ollama

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import TfidfVectorizer


# CONFIGURACIÓN


MODEL_NAME      = "llama3.1"
N_PER_PARK      = 200
EXPLORE_N       = 500
N_TOPICS        = 6
TOP_N_WORDS     = 10
RANDOM_SEED     = 42
OUTPUT_CSV      = "../Topic/Resultados/Topico/LLM_results.csv"
OUTPUT_TOPICS   = "../Topic/Resultados/Topico/LLM_topics_discovered.json"


# 1. MUESTREO — 200 por parque, aleatorio

def stratified_sample(df: pd.DataFrame,
                      text_col: str  = "text",
                      park_col: str  = "nombre_parque",
                      n_per_park: int = N_PER_PARK,
                      seed: int      = RANDOM_SEED) -> pd.DataFrame:

    sampled = (df.groupby(park_col, group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), n_per_park), random_state=seed))
                 .reset_index(drop=True))

    print(f"Muestra final: {len(sampled)} reseñas de {sampled[park_col].nunique()} parques")
    print(sampled[park_col].value_counts().to_string())
    return sampled



# 2. FASE 1 — DESCUBRIMIENTO DE TÓPICOS


def discover_topics(df_sample: pd.DataFrame,
                    text_col: str  = "text",
                    n_explore: int = EXPLORE_N,
                    n_topics: int  = N_TOPICS,
                    model: str     = MODEL_NAME) -> dict:

    sample_texts = (df_sample.sample(n=min(n_explore, len(df_sample)), random_state=RANDOM_SEED)
                              [text_col].tolist())

    reviews_str = "\n".join([f"- {str(r)[:200]}" for r in sample_texts])

    prompt = f"""Sos un investigador analizando reseñas de parques públicos de Montevideo, Uruguay.

A continuación hay {len(sample_texts)} reseñas reales de usuarios:

{reviews_str}

Tu tarea:
1. Identificar exactamente {n_topics} temas principales que emergen de estas reseñas
2. Cada tema debe ser distinto y representar una dimensión real de lo que habla la gente
3. Basate SOLO en lo que ves en las reseñas, no en categorías predefinidas

Respondé ÚNICAMENTE con un objeto JSON válido, sin texto adicional:
{{
  "topics": [
    {{"id": 0, "name": "<nombre corto>", "description": "<descripción de 1 oración>", "example_keywords": ["palabra1", "palabra2", "palabra3"]}},
    {{"id": 1, "name": "<nombre corto>", "description": "<descripción de 1 oración>", "example_keywords": ["palabra1", "palabra2", "palabra3"]}},
    {{"id": 2, "name": "<nombre corto>", "description": "<descripción de 1 oración>", "example_keywords": ["palabra1", "palabra2", "palabra3"]}},
    {{"id": 3, "name": "<nombre corto>", "description": "<descripción de 1 oración>", "example_keywords": ["palabra1", "palabra2", "palabra3"]}},
    {{"id": 4, "name": "<nombre corto>", "description": "<descripción de 1 oración>", "example_keywords": ["palabra1", "palabra2", "palabra3"]}},
    {{"id": 5, "name": "<nombre corto>", "description": "<descripción de 1 oración>", "example_keywords": ["palabra1", "palabra2", "palabra3"]}}
  ]
}}"""

    print(f"Fase 1: descubriendo tópicos con {len(sample_texts)} reseñas...")

    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.3, "num_predict": 800}
        )
        raw = response["message"]["content"].strip()
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            data   = json.loads(json_match.group())
            topics = {t["id"]: t for t in data["topics"]}

            print("Tópicos descubiertos:")
            for tid, t in topics.items():
                print(f"  [{tid}] {t['name']}: {t['description']}")
                print(f"       Keywords: {', '.join(t['example_keywords'])}")
            return topics

    except Exception as e:
        print(f"Error en descubrimiento: {e}")
        print(f"Respuesta cruda: {raw[:400]}")
        return {}


def validate_topics_interactive(topics: dict) -> dict:
    print("Tópicos descubiertos:")
    for tid, t in topics.items():
        print(f"  [{tid}] {t['name']}: {t['description']}")
    return topics



# 3. FASE 2 — CLASIFICACIÓN


def build_classification_prompt(review_text: str, topics: dict) -> str:
    topics_str = "\n".join(
        [f"  {t['id']}: {t['name']} — {t['description']}" for t in topics.values()]
    )
    max_id = max(topics.keys())
    return f"""Clasificá la siguiente reseña de un parque público de Montevideo en UNO de estos tópicos:

{topics_str}

Reseña:
\"\"\"{review_text}\"\"\"

Respondé ÚNICAMENTE con JSON válido, sin texto adicional:
{{
  "topic_id": <número entre 0 y {max_id}>,
  "confidence": <número entre 0.0 y 1.0>,
  "reason": "<explicación muy breve>"
}}"""


def classify_review(review_text: str, topics: dict,
                    model: str = MODEL_NAME,
                    max_retries: int = 3) -> dict:

    if not review_text or len(str(review_text).strip()) < 5:
        return {"topic_id": -1, "confidence": 0.0, "reason": "vacía"}

    prompt = build_classification_prompt(str(review_text)[:400], topics)

    for attempt in range(max_retries):
        try:
            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0.1, "num_predict": 120}
            )
            raw = response["message"]["content"].strip()
            json_match = re.search(r'\{.*?\}', raw, re.DOTALL)
            if json_match:
                result     = json.loads(json_match.group())
                topic_id   = int(result.get("topic_id", -1))
                confidence = float(result.get("confidence", 0.5))
                reason     = result.get("reason", "")

                if topic_id not in topics:
                    topic_id = -1
                confidence = max(0.0, min(1.0, confidence))

                return {"topic_id": topic_id, "confidence": confidence, "reason": reason}

        except (json.JSONDecodeError, Exception):
            time.sleep(1)

    return {"topic_id": -1, "confidence": 0.0, "reason": "error tras reintentos"}


def classify_corpus(df_sample: pd.DataFrame, topics: dict,
                    text_col: str = "text",
                    model: str    = MODEL_NAME,
                    checkpoint_every: int = 50) -> pd.DataFrame:

    results = []
    total   = len(df_sample)
    start   = time.time()

    print(f"Fase 2: clasificando {total} reseñas con {model}...")

    for i, (idx, row) in enumerate(tqdm(df_sample.iterrows(), total=total)):
        res = classify_review(str(row[text_col]), topics, model)
        res["original_idx"]    = idx
        res["text"]            = row[text_col]
        res["nombre_parque"]   = row.get("nombre_parque", "")
        results.append(res)

        if (i + 1) % checkpoint_every == 0:
            elapsed = time.time() - start
            eta_min = (elapsed / (i + 1)) * (total - i - 1) / 60
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            print(f"  Checkpoint {i+1}/{total} — ETA: {eta_min:.0f} min")

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Clasificación completa → {OUTPUT_CSV}")
    return results_df



# 4. MÉTRICAS


def extract_topic_words(results_df: pd.DataFrame, topics: dict,
                        text_col: str  = "text",
                        topic_col: str = "topic_id",
                        top_n: int     = TOP_N_WORDS) -> dict:
    topic_words = {}
    for tid in sorted(topics.keys()):
        texts = results_df[results_df[topic_col] == tid][text_col].tolist()
        if len(texts) < 3:
            topic_words[tid] = []
            continue
        try:
            tfidf  = TfidfVectorizer(max_features=200, min_df=2, strip_accents='unicode')
            matrix = tfidf.fit_transform(texts).toarray()
            names  = tfidf.get_feature_names_out()
            top_i  = matrix.mean(axis=0).argsort()[-top_n:][::-1]
            topic_words[tid] = [names[i] for i in top_i]
        except Exception:
            topic_words[tid] = []
    return topic_words


def calculate_coherence_cv(results_df: pd.DataFrame, topics: dict,
                            text_col: str  = "text",
                            topic_col: str = "topic_id") -> dict:
    print("\n--- Coherencia C_v ---")
    tokenized  = [str(t).lower().split() for t in results_df[text_col]]
    dictionary = Dictionary(tokenized)
    topic_words = extract_topic_words(results_df, topics, text_col, topic_col)

    valid = {k: v for k, v in topic_words.items() if len(v) >= 3}
    if not valid:
        print("  No hay tópicos válidos para C_v")
        return {}

    cm        = CoherenceModel(topics=list(valid.values()),
                               texts=tokenized, dictionary=dictionary, coherence='c_v')
    global_cv = cm.get_coherence()
    per_topic = dict(zip(valid.keys(), cm.get_coherence_per_topic()))

    print(f"  C_v global: {global_cv:.4f}")
    for tid, score in per_topic.items():
        print(f"  [{tid}] {topics[tid]['name']}: {score:.4f}")

    return {"global_cv": global_cv, "per_topic": per_topic, "topic_words": topic_words}


def distribution_report(results_df: pd.DataFrame, topics: dict,
                        topic_col: str = "topic_id") -> pd.DataFrame:
    total    = len(results_df)
    outliers = (results_df[topic_col] == -1).sum()
    print(f"\n--- Distribución ---")
    print(f"Total: {total}  |  Outliers: {outliers} ({outliers/total*100:.1f}%)\n")

    rows = []
    for tid in sorted(results_df[topic_col].unique()):
        count    = (results_df[topic_col] == tid).sum()
        pct      = count / total * 100
        avg_conf = results_df[results_df[topic_col] == tid]["confidence"].mean()
        label    = "Outlier" if tid == -1 else topics.get(tid, {}).get("name", f"T{tid}")
        print(f"  [{tid:2d}] {label:<35} {count:5d}  ({pct:5.1f}%)  conf={avg_conf:.3f}")
        rows.append({"topic_id": tid, "name": label, "count": count,
                     "pct": round(pct, 2), "avg_confidence": round(avg_conf, 3)})

    return pd.DataFrame(rows)



# 5. PIPELINE COMPLETO


def run_llm_pipeline(df: pd.DataFrame,
                     text_col: str   = "text",
                     park_col: str   = "nombre_parque",
                     model: str      = MODEL_NAME,
                     n_per_park: int = N_PER_PARK,
                     resume_from: str = None) -> dict:

    # Muestreo
    df_sample = stratified_sample(df, text_col, park_col, n_per_park)

    # Fase 1: descubrir tópicos
    topics = discover_topics(df_sample, text_col, model=model)
    if not topics:
        raise RuntimeError("No se pudieron descubrir tópicos. ¿Está Ollama corriendo?")

    # Validación interactiva
    topics = validate_topics_interactive(topics)

    # Guardar tópicos descubiertos
    with open(OUTPUT_TOPICS, "w", encoding="utf-8") as f:
        json.dump(topics, f, ensure_ascii=False, indent=2)
    print(f"Tópicos guardados → {OUTPUT_TOPICS}")

    # Fase 2: clasificar (con opción de retomar checkpoint)
    if resume_from and os.path.exists(resume_from):
        print(f"\nRetomando desde: {resume_from}")
        results_df     = pd.read_csv(resume_from)
        classified_idx = set(results_df["original_idx"])
        pending        = df_sample[~df_sample.index.isin(classified_idx)]
        if len(pending) > 0:
            new_results = classify_corpus(pending, topics, text_col, model)
            results_df  = pd.concat([results_df, new_results], ignore_index=True)
    else:
        results_df = classify_corpus(df_sample, topics, text_col, model)

    # Métricas
    dist      = distribution_report(results_df, topics)
    coherence = calculate_coherence_cv(results_df, topics, text_col)

    summary = {
        "model":        model,
        "sample_size":  len(results_df),
        "n_per_park":   n_per_park,
        "topics":       topics,
        "distribution": dist,
        "coherence_cv": coherence.get("global_cv"),
        "per_topic_cv": coherence.get("per_topic", {}),
        "topic_words":  coherence.get("topic_words", {}),
        "results_df":   results_df,
    }

    print(f"\n{'='*50}")
    print(f"LLM ({model})  |  {len(results_df)} reseñas  |  {n_per_park} por parque")
    if summary["coherence_cv"]:
        print(f"Coherencia C_v: {summary['coherence_cv']:.4f}")
    print(f"Tópicos: {[t['name'] for t in topics.values()]}")
    print(f"{'='*50}")

    return summary


# 6. USO


if __name__ == "__main__":

    df_generic_cleaned = pd.read_pickle("../df_generic_cleaned.pkl")

    # Prueba
    # summary = run_llm_pipeline(df_generic_cleaned, n_per_park=10)

    # 200 por parque = 2000 reseñas (~2-3 hs)
    summary = run_llm_pipeline(df_generic_cleaned, n_per_park=200)

    # Si se cortó, retomar desde el checkpoint
    # summary = run_llm_pipeline(df_generic_cleaned, n_per_park=200,
    #                            resume_from="llm_results.csv")

### Identificador de cada reseña clasificada

In [ ]:
"""
Mapeo post-hoc del LLM: agrega columna 'id' al CSV de resultados
=================================================================
Reconstruye el stratified_sample con random_state=42 (determinista) y
mapea cada original_idx (0..1999) al id real del df base.

NO re-corre Llama. Toma segundos.
"""

import pandas as pd

# ---------- CONFIGURACIÓN ----------
BASE = ".."

PATH_DF_BASE     = f"{BASE}/df_generic_cleaned.pkl"
PATH_LLM_INPUT   = f"{BASE}/Topic/Resultados/Topico/LLM_results.csv"
PATH_LLM_OUTPUT  = f"{BASE}/Topic/Resultados/Topico/LLM_results_with_id.csv"

SEED       = 42
N_PER_PARK = 200
PARK_COL   = 'nombre_parque'


# ---------- 1. CARGAR DF BASE ----------
df = pd.read_pickle(PATH_DF_BASE)
assert 'id' in df.columns, "El df base no tiene columna 'id'"
print(f"df base: {len(df)} reseñas, columnas {list(df.columns)}")


# ---------- 2. RECONSTRUIR EL SAMPLE ----------
# Misma lógica EXACTA que stratified_sample del script LLM original.
df_sample = (df.groupby(PARK_COL, group_keys=False)
               .apply(lambda g: g.sample(min(len(g), N_PER_PARK),
                                         random_state=SEED))
               .reset_index(drop=True))

print(f"\nSample reconstruido: {len(df_sample)} reseñas")
print(f"Por parque:\n{df_sample[PARK_COL].value_counts().to_string()}")


# ---------- 3. CARGAR RESULTADOS DEL LLM ----------
llm = pd.read_csv(PATH_LLM_INPUT)
print(f"\nCSV LLM: {len(llm)} filas")
print(f"original_idx: {llm['original_idx'].min()} a {llm['original_idx'].max()}")


# ---------- 4. VERIFICACIÓN DEFENSIVA ----------
# Confirmar que el texto del LLM en original_idx N coincide con el del
# df_sample en posición N. Si no, el mapping no es válido.
n_check = min(50, len(llm))
matches = 0
for _, row in llm.head(n_check).iterrows():
    idx = int(row['original_idx'])
    if idx < len(df_sample):
        texto_sample = str(df_sample.iloc[idx]['text']).strip()
        texto_llm    = str(row['text']).strip()
        if texto_sample == texto_llm:
            matches += 1

print(f"\nVerificación texto-a-texto en {n_check} filas: {matches}/{n_check} coinciden")
assert matches == n_check, (
    f"Los textos no coinciden ({matches}/{n_check}). "
    "El sample reconstruido difiere del usado por el LLM. "
    "Posibles causas: el pickle base cambió de orden, o random_state distinto."
)


# ---------- 5. MAPEAR original_idx → id ----------
idx_to_id = df_sample['id'].to_dict()   # {0: id_de_la_fila_0, 1: ...}
llm['id'] = llm['original_idx'].map(idx_to_id)

assert llm['id'].notna().all(), "Quedaron original_idx sin mapear"
print(f"\nMapeo exitoso. Rango de id mapeado: {llm['id'].min()} a {llm['id'].max()}")


# ---------- 6. GUARDAR ----------
# Reordenar columnas para que id aparezca primero
cols = ['id', 'original_idx', 'nombre_parque', 'topic_id',
        'confidence', 'reason', 'text']
llm = llm[cols]
llm.to_csv(PATH_LLM_OUTPUT, index=False)

print(f"\nGuardado: {PATH_LLM_OUTPUT}")
print(llm.head())

### Coherencia C_v

In [ ]:
# RECÁLCULO DE COHERENCIA C_v DEL LLM CON STOPWORDS FILTRADAS

import unicodedata
import pandas as pd
from nltk.corpus import stopwords as nltk_sw
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary


def quitar_tildes(t):
    return ''.join(c for c in unicodedata.normalize('NFKD', t)
                   if unicodedata.category(c) != 'Mn')


# 1. CARGA
results = pd.read_csv(
    "../Topic/Resultados/Topico/LLM_results.csv"
)
df_orig = pd.read_pickle(
    "../df_generic_cleaned.pkl"
)
print(f"Reseñas clasificadas: {len(results)}")


# 2. STOPWORDS — mismas capas que BERTopic / W2V / LDA corregidos
stop_es = set(nltk_sw.words('spanish'))

stopwords_contexto = {
    'lugar', 'lugares', 'sitio', 'sitios', 'espacio', 'espacios',
    'zona', 'zonas', 'parte', 'partes', 'parque', 'parques',
    'montevideo', 'uruguay',
    'visita', 'visitar', 'visite', 'visitamos', 'visitarlo', 'visitando',
    'paseo', 'pasear', 'paseando', 'paseos',
    'ir', 'fui', 'fue', 'fuimos', 'ido', 'vamos', 'van', 'vas',
    'hacer', 'hice', 'hicimos', 'hace', 'hacen',
    'dia', 'días', 'dias', 'noche', 'noches',
    'fin', 'semana', 'semanas', 'año', 'años', 'mes', 'meses',
    'vez', 'veces', 'momento', 'momentos',
    'cosa', 'cosas', 'caso', 'tema', 'tipo', 'tipos',
    'aunque', 'sino', 'pues', 'tal', 'tan', 'asi', 'así',
    'ahi', 'ahí', 'alla', 'allá', 'aca', 'acá',
    'pais', 'país', 'ciudad', 'capital',
    'gente', 'personas', 'persona', 'publico', 'público',
    'mate', 'pasar', 'pasamos', 'pasando',
    'puede', 'pueden', 'poder', 'podemos',
    'tiene', 'tienen', 'tener', 'tenemos',
    'está', 'esta', 'están', 'estan', 'estar', 'estamos',
    'hay', 'haber', 'había', 'habia',
    'ser', 'siendo', 'sido',
    'todo', 'toda', 'todos', 'todas',
    'uno', 'una', 'unos', 'unas',
    'otro', 'otra', 'otros', 'otras',
    'donde', 'cuando', 'como', 'cómo',
    'para', 'por', 'con', 'sin', 'desde', 'hasta',
    'más', 'mas', 'menos', 'entre',
    'cada', 'algo', 'nada', 'siempre', 'nunca', 'también', 'tambien',
    'ya', 'aún', 'aun', 'aquí', 'aqui',
    'dar', 'dio', 'dan', 'dieron',
    'ver', 'veo', 'vimos', 'visto',
    'decir', 'dice', 'dicen',
    'saber', 'sabemos',
    'quedar', 'queda', 'quedan',
    'llegar', 'llega', 'llegamos',
    'salir', 'sale', 'salimos',
    'solo', 'sólo', 'solamente',
    'prado', 'gracias', 'gostei',
}

stopwords_sentimiento = {
    'lindo', 'linda', 'lindos', 'lindas', 'lindísimo', 'lindísima',
    'hermoso', 'hermosa', 'hermosos', 'hermosas',
    'bonito', 'bonita', 'bonitos', 'bonitas',
    'bello', 'bella', 'bellos', 'bellas',
    'precioso', 'preciosa', 'preciosos', 'preciosas',
    'divino', 'divina', 'divinos', 'divinas',
    'impecable', 'impecables',
    'maravilloso', 'maravillosa', 'maravillosos', 'maravillosas',
    'excelente', 'excelentes', 'espectacular', 'espectaculares',
    'genial', 'geniales', 'fantástico', 'fantastico', 'fantástica', 'fantastica',
    'increíble', 'increible', 'increíbles', 'increibles',
    'impresionante', 'impresionantes',
    'terrible', 'terribles',
    'feo', 'fea', 'feos', 'feas',
    'horrible', 'horribles', 'espantoso', 'espantosa',
    'pésimo', 'pesimo', 'pésima', 'pesima',
    'lamentable', 'lamentables',
    'desastre', 'desastroso', 'desastrosa',
    'patético', 'patetico', 'patética', 'patetica',
    'bueno', 'buena', 'buenos', 'buenas', 'buen',
    'malo', 'mala', 'malos', 'malas', 'mal',
    'mejor', 'mejores', 'peor', 'peores',
    'mayor', 'mayores', 'menor', 'menores',
    'gran', 'grande', 'grandes',
    'bien', 'super', 'súper', 're', 'muy', 'bastante',
    'poco', 'poca', 'pocos', 'pocas',
    'mucho', 'mucha', 'muchos', 'muchas',
    'agradable', 'agradables', 'desagradable', 'desagradables',
    'recomendable', 'recomendado', 'recomendada',
    'tranquilo', 'tranquila', 'tranquilos', 'tranquilas',
    'paz', 'relax', 'relajante',
    'gusto', 'gustó', 'gustar', 'gusta', 'gustan',
    'encanto', 'encantó', 'encantar', 'encanta', 'encantan',
    'disfrutar', 'disfrute', 'disfruté', 'disfruta', 'disfrutamos',
    'recomendar', 'recomiendo', 'recomendamos',
    'pena', 'vale', 'valer',
    'ideal', 'ideales', 'perfecto', 'perfecta', 'perfectos', 'perfectas',
    'destacar', 'destaca', 'destacan',
    'disfrutable', 'formoso', 'belo', 'feio', 'horrivel', 'terrivel', 'bom', 'boa',
    'mau', 'ruim', 'espetacular', 'incrivel', 'maravilhoso',
    'impecavel', 'impressionante', 'pessimo', 'lamentavel',
    'melhor', 'pior', 'maior', 'bem', 'muito', 'pouco', 'agradavel',
    'desagradavel', 'recomendavel', 'gostar', 'desfrutar', 'aproveitar',
    'perfeito',
}

stopwords_ingles = {
    'the', 'and', 'is', 'it', 'to', 'of', 'in', 'for', 'on', 'with',
    'that', 'this', 'was', 'are', 'be', 'at', 'an', 'or', 'not', 'but',
    'very', 'nice', 'good', 'great', 'beautiful', 'bad', 'best', 'place',
    'park', 'visit', 'really', 'love', 'like', 'amazing', 'wonderful',
    'been', 'have', 'has', 'had', 'from', 'there', 'they', 'will', 'can',
    'you', 'your', 'we', 'our', 'my', 'just', 'would', 'about', 'what',
    'one', 'all', 'also', 'some', 'so', 'if', 'more', 'when', 'which',
}

stopwords_typos = {
    'exelente', 'exelentes', 'recomeir', 'recomiendable',
    'imprecionante', 'ermoso', 'ermosa', 'escelente',
    'recomendacion', 'lindar', 'mas',
}

stopwords_nombre_parque = set()
for nombre in df_orig['nombre_parque'].str.lower().unique():
    for palabra in nombre.split():
        stopwords_nombre_parque.add(palabra)

all_stopwords = sorted({
    quitar_tildes(w) for w in (
        stop_es | stopwords_contexto | stopwords_sentimiento
        | stopwords_ingles | stopwords_typos | stopwords_nombre_parque
    )
})
print(f"Stopwords totales: {len(all_stopwords)}")


# 3. RE-EXTRAER TOP WORDS POR TÓPICO CON STOPWORDS FILTRADAS
def extraer_top_words(results_df, stopwords, top_n=10):
    topic_words = {}
    for tid in sorted(results_df['topic_id'].unique()):
        if tid == -1:
            continue
        texts = results_df[results_df['topic_id'] == tid]['text'].astype(str).tolist()
        if len(texts) < 3:
            topic_words[tid] = []
            continue
        try:
            tfidf = TfidfVectorizer(
                max_features=200,
                min_df=2,
                stop_words=stopwords,
                strip_accents='unicode'
            )
            matrix = tfidf.fit_transform(texts).toarray()
            names = tfidf.get_feature_names_out()
            top_i = matrix.mean(axis=0).argsort()[-top_n:][::-1]
            topic_words[tid] = [names[i] for i in top_i]
        except Exception as e:
            print(f"  Tópico {tid} error: {e}")
            topic_words[tid] = []
    return topic_words

topic_words = extraer_top_words(results, all_stopwords)

print("\nTop words por tópico (con stopwords filtradas):")
for tid, words in topic_words.items():
    print(f"  [{tid}]: {words}")


# 4. RECALCULAR C_v
tokenized_docs = [
    [quitar_tildes(t) for t in str(doc).lower().split()]
    for doc in results['text']
]
dictionary = Dictionary(tokenized_docs)

valid = {k: v for k, v in topic_words.items() if len(v) >= 3}

cm = CoherenceModel(
    topics=list(valid.values()),
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
coh_global = cm.get_coherence()
per_topic = dict(zip(valid.keys(), cm.get_coherence_per_topic()))

print(f"\nCoherencia C_v global (con stopwords filtradas): {coh_global:.4f}")
print(f"(Referencia: original sin filtrar = 0.6137)")
for tid, score in per_topic.items():
    print(f"  Tópico {tid}: {score:.4f}")

### Coherencia C_v con el diccionario de exclusión

In [ ]:
# === Coherencia C_v — LLM (autocontenido, con stopwords nuevas) ===
import unicodedata
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from nltk.corpus import stopwords as nltk_stop

# --- RUTAS: apuntan al CSV que recuperé y al df base ---
PATH_RESULTS = "../Topic/Resultados/Topico/LLM_results.csv"
PATH_DF      = "../df_generic_cleaned.pkl"

results = pd.read_csv(PATH_RESULTS)
df_base = pd.read_pickle(PATH_DF)
print(f"Reseñas clasificadas: {len(results)}")

def remove_accents(t):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(t))
                   if unicodedata.category(c) != 'Mn')

# --- MISMAS stopwords que los otros notebooks ---
CONTEXTO = {
    'lugar','sitio','espacio','zona','parte','parque','montevideo','uruguay',
    'visita','visitar','paseo','pasear','ir','hacer','dia','noche','fin','semana',
    'año','vez','momento','cosa','caso','tema','tipo','aunque','sino','pues','tal',
    'tan','asi','ahi','alla','aca','pais','ciudad','capital','gente','persona',
    'publico','mate','pasar','quedar','estar','ser','haber','tener','poder','deber',
    'dar','ver','saber','querer','llegar','decir','salir','venir','encontrar',
    'dejar','poner','parecer','llevar','seguir','etc','siempre','alguno','solo',
    'medio','ahora','mismo',
}
SENTIMIENTO = {
    'lindo','hermoso','bonito','bello','precioso','feo','horrible','espantoso',
    'terrible','bueno','malo','buen','excelente','espectacular','genial',
    'fantastico','increible','maravilloso','divino','impecable','impresionante',
    'pesimo','lamentable','desastre','patetico','mejor','peor','mayor','menor',
    'gran','grande','bien','mal','super','muy','bastante','poco','mucho','agradable',
    'desagradable','recomendable','recomendado','tranquilo','paz','relax','gustar',
    'encantar','disfrutar','recomendar','valer','ideal','perfecto','destacar','pena',
    'disfrutable',
}
ERRORES = {
    'lindar','exelente','exelentes','escelente','imprecionante','ermoso','ermosa',
    'recomeir','recomiendable','recomendacion','mas',
}
CORTESIA = {'gracias'}
LUGARES  = {'prado'}
PROTEGER = set()

nombres_parque = set()
for nombre in df_base['nombre_parque'].str.lower().unique():
    for token in nombre.split():
        nombres_parque.add(remove_accents(token))
nombres_parque -= PROTEGER

final_stopwords = {remove_accents(w) for w in nltk_stop.words('spanish')}
for grupo in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in grupo}
final_stopwords |= nombres_parque
final_stopwords = sorted(final_stopwords)
print(f"Stopwords totales: {len(final_stopwords)}")

# --- Lematizar (mismo preprocesamiento que los otros) ---
nlp = spacy.load('es_core_news_sm', disable=['parser','ner'])
results['texto_lematizado'] = [
    ' '.join(t.lemma_ for t in doc if t.is_alpha)
    for doc in nlp.pipe(results['text'].astype(str).tolist(), batch_size=200)
]

# --- Top words por topico + C_v ---
TOP_N = 15
topic_words = []
for tid in sorted(results['topic_id'].unique()):
    if tid == -1:
        continue
    textos = results[results['topic_id'] == tid]['texto_lematizado'].tolist()
    if len([t for t in textos if t.strip()]) < 3:
        continue
    tfidf = TfidfVectorizer(max_features=200, min_df=2,
                            stop_words=final_stopwords, strip_accents='unicode')
    M = tfidf.fit_transform(textos).toarray()
    nombres = tfidf.get_feature_names_out()
    idx = M.mean(axis=0).argsort()[-TOP_N:][::-1]
    topic_words.append([nombres[i] for i in idx])

textos_tokenizados = [remove_accents(t).split() for t in results['texto_lematizado']]
diccionario = Dictionary(textos_tokenizados)

cm = CoherenceModel(topics=topic_words, texts=textos_tokenizados,
                    dictionary=diccionario, coherence='c_v')
print(f"\nCoherencia C_v (LLM): {cm.get_coherence():.4f}")
for i, s in enumerate(cm.get_coherence_per_topic()):
    print(f"  Topico {i}: {s:.4f}")